# Third-Place Match Behavioral Analysis

Objective: build a reusable pipeline to collect and clean player-level match data for England and France, ending with an analysis-ready cleaned dataframe.

This notebook covers only:
1. Setup and imports
2. Data collection from Sofascore
3. Raw data assembly
4. Data quality checks
5. Data cleaning and standardization

No modeling or comparisons are coded yet.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from ScraperFC.sofascore import Sofascore

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1) Define Reusable Scraping Helpers

This function enriches player stats with match metadata so each row stays at player-match grain.

In [2]:
sf = Sofascore()

def scrape_matches(match_ids):
    rows = []
    for match_id in match_ids:
        try:
            meta = sf.get_match_dict(match_id)
            df = sf.scrape_player_match_stats(match_id)

            if df is None or df.empty:
                print(f"No rows returned for match_id={match_id}")
                continue

            df = df.copy()
            df["match_id"] = match_id
            df["home_team"] = meta.get("homeTeam", {}).get("name")
            df["away_team"] = meta.get("awayTeam", {}).get("name")
            df["home_score"] = meta.get("homeScore", {}).get("current")
            df["away_score"] = meta.get("awayScore", {}).get("current")
            df["tournament"] = meta.get("tournament", {}).get("name")
            df["season"] = meta.get("season", {}).get("name")
            df["status"] = meta.get("status", {}).get("description")
            df["start_time"] = meta.get("startTimestamp")

            rows.append(df)
        except Exception as e:
            print(f"match_id={match_id} failed: {e}")

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

## 2) Declare Match IDs and Collect Raw Data

Match IDs are centralized so the workflow is reusable and easy to audit.

In [ ]:
# Legacy explicit per-match scraping block (kept as a separate cell)
# England matches
eng_cro_df = scrape_matches([15186504])  # England vs Croatia
eng_pan_df = scrape_matches([15186676])  # England vs Panama
eng_gha_df = scrape_matches([15186672])  # England vs Ghana
eng_con_32_df = scrape_matches([12813020])  # England vs DR Congo (R32)
eng_mex_16_df = scrape_matches([12813007])  # England vs Mexico (R16)
eng_nor_8_df = scrape_matches([12813017])  # England vs Norway (QF)
eng_arg_4_df = scrape_matches([12812996])  # England vs Argentina (SF)

# France matches
fra_nor_df = scrape_matches([15186537])  # France vs Norway
fra_sen_df = scrape_matches([15186501])  # France vs Senegal
fra_ira_df = scrape_matches([15186769])  # France vs Iraq
fra_swe_32_df = scrape_matches([12812995])  # France vs Sweden (R32)
fra_par_16_df = scrape_matches([12813010])  # France vs Paraguay (R16)
fra_mor_8_df = scrape_matches([12813016])  # France vs Morocco (QF)
fra_spa_4_df = scrape_matches([12813008])  # France vs Spain (SF)

third_place_df = scrape_matches([12813003])  # Third-place match

In [3]:
match_map = {
    # England matches
    "england_vs_croatia": 15186504,
    "england_vs_panama": 15186676,
    "england_vs_ghana": 15186672,
    "england_vs_dr_congo_r32": 12813020,
    "england_vs_mexico_r16": 12813007,
    "england_vs_norway_qf": 12813017,
    "england_vs_argentina_sf": 12812996,

    # France matches
    "france_vs_norway": 15186537,
    "france_vs_senegal": 15186501,
    "france_vs_iraq": 15186769,
    "france_vs_sweden_r32": 12812995,
    "france_vs_paraguay_r16": 12813010,
    "france_vs_morocco_qf": 12813016,
    "france_vs_spain_sf": 12813008,

    # Third-place match
    "third_place": 12813003,
}

all_match_ids = list(match_map.values())
raw_df = scrape_matches(all_match_ids)

# Fallback to local CSVs if scraping returns empty (e.g., API/session issues).
if raw_df.empty:
    fallback_files = [
        Path("england_matches.csv"),
        Path("sofascore_third_place_player_stats.csv"),
    ]
    fallback_dfs = [pd.read_csv(p) for p in fallback_files if p.exists()]
    raw_df = pd.concat(fallback_dfs, ignore_index=True) if fallback_dfs else pd.DataFrame()

print("raw_df shape:", raw_df.shape)
raw_df.head(3)

Running


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

## 3) Raw Data Quality Checks

Before cleaning, profile shape, duplicate columns, null rates, and key field coverage.

In [ ]:
if raw_df.empty:
    raise ValueError("raw_df is empty. Scraping and fallback CSV loading both returned no data.")

print("Rows, Columns:", raw_df.shape)
print("Duplicate column names:", int(raw_df.columns.duplicated().sum()))

null_pct = (raw_df.isna().mean() * 100).sort_values(ascending=False)
display(null_pct.head(15).to_frame("null_pct_top15"))

key_cols = ["name", "teamName", "match_id", "minutesPlayed", "start_time"]
existing_key_cols = [c for c in key_cols if c in raw_df.columns]
display(raw_df[existing_key_cols].head(5))

## 4) Clean and Standardize Data

Cleaning steps follow the README philosophy: normalize names, fix types, remove noisy metadata, and preserve an analysis-ready player-match table.

In [ ]:
def to_snake_case(col):
    col = str(col).strip()
    col = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", col)
    col = col.replace(" ", "_").replace("-", "_").replace("/", "_")
    col = re.sub(r"[^0-9a-zA-Z_]", "", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col.lower()

clean_df = raw_df.copy()

# 1) Normalize column names
clean_df.columns = [to_snake_case(c) for c in clean_df.columns]

# 2) Remove duplicate columns that can appear after scraping/concatenation
clean_df = clean_df.loc[:, ~clean_df.columns.duplicated()].copy()

# 3) Drop low-value metadata/technical fields
drop_cols = [
    "field_translations",
    "country",
    "rating_versions",
    "statistics_type",
    "proposed_market_value_raw",
    "market_value_currency",
    "date_of_birth_timestamp",
    "slug",
    "sofascore_id",
    "id",
    "user_count",
    "gender",
]
clean_df = clean_df.drop(columns=[c for c in drop_cols if c in clean_df.columns], errors="ignore")

# 4) Parse booleans
for bool_col in ["substitute", "captain"]:
    if bool_col in clean_df.columns:
        clean_df[bool_col] = (
            clean_df[bool_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False})
        )

# 5) Parse timestamps
if "start_time" in clean_df.columns:
    clean_df["start_time"] = pd.to_datetime(clean_df["start_time"], unit="s", errors="coerce")

# 6) Numeric conversion for likely metric columns
non_numeric_candidates = {
    "name", "first_name", "last_name", "short_name", "position", "team_name",
    "home_team", "away_team", "tournament", "season", "status", "captain",
    "substitute",
}
numeric_cols = [
    c for c in clean_df.columns
    if c not in non_numeric_candidates and c != "start_time"
]
for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="ignore")

# 7) Remove exact duplicate rows
before_rows = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
after_rows = len(clean_df)

# 8) Basic null-handling rule for minutes played
if "minutes_played" in clean_df.columns:
    clean_df["minutes_played"] = clean_df["minutes_played"].fillna(0)

print("Rows before dedup:", before_rows)
print("Rows after dedup:", after_rows)
print("clean_df shape:", clean_df.shape)

display(clean_df.head(5))

## 5) Next Section Titles (No Code Yet)

Use these as the next notebook headings:

1. Variable Selection for Behavioral Dimensions
2. Player Inclusion Rules and Minutes Thresholds
3. Per-90 Feature Engineering
4. Baseline Construction (Pre-Semifinal)
5. Baseline vs Semifinal Comparison
6. Baseline vs Third-Place Comparison
7. France vs England Team-Level Summary
8. Visualization: Behavioral Profiles by Match Context
9. Interpretation and Practical Takeaways
10. Limitations and Data Quality Caveats